In [ ]:
# ============================================
# 06_Evaluation.ipynb
# EVALUASI DAN PERBANDINGAN MODEL
# DETEKSI KANTUK PENGEMUDI
# ============================================

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from matplotlib.lines import Line2D

# ============================================
# SETUP
# ============================================
warnings.filterwarnings('ignore')

print("="*60)
print("📊 EVALUASI DAN PERBANDINGAN MODEL")
print("="*60)

# ============================================
# 1. SETUP PATHS
# ============================================

project_root = Path.cwd().parent if Path.cwd().name in ['notebooks', 'notebook'] else Path.cwd()
results_dir = project_root / "results"
comparison_dir = results_dir / "comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)

print(f"\n[Paths]")
print(f"   - Project: {project_root}")
print(f"   - Results: {results_dir}")
print(f"   - Comparison: {comparison_dir}")

# ============================================
# 2. LOAD RESULTS DARI MODEL
# ============================================

print("\n[Loading Model Results]")

model_results = {}
model_names = ['cnn', 'mobilenetv2', 'yolo']
model_labels = {'cnn': 'CNN', 'mobilenetv2': 'MobileNetV2', 'yolo': 'YOLOv8'}
model_sizes = {'cnn': 125, 'mobilenetv2': 14, 'yolo': 6}
model_colors = {'cnn': '#ff6b6b', 'mobilenetv2': '#4ecdc4', 'yolo': '#45b7d1'}

available_models = []

for model_name in model_names:
    result_file = results_dir / model_name / 'results.json'
    if result_file.exists():
        try:
            with open(result_file, 'r') as f:
                model_results[model_name] = json.load(f)
            print(f"   ✅ Loaded {model_labels[model_name]} results")
            available_models.append(model_name)
        except Exception as e:
            print(f"   ⚠️ Error loading {model_labels[model_name]}: {e}")
            model_results[model_name] = None
    else:
        print(f"   ⚠️ {model_labels[model_name]} results not found (skip)")
        model_results[model_name] = None

if len(available_models) == 0:
    print("\n❌ Tidak ada hasil model yang ditemukan!")
    print("   Pastikan training sudah selesai untuk setidaknya satu model.")
    raise SystemExit("Tidak ada data untuk dievaluasi")

print(f"\n✅ Model tersedia: {[model_labels[m] for m in available_models]}")

# ============================================
# 3. METRIK PERBANDINGAN
# ============================================

print("\n[Comparison Metrics]")

# Extract metrics from each model
comparison_data = {
    'Model': [],
    'Accuracy': [],
    'Precision': [],
    'Recall': [],
    'F1-Score': [], 
    'AUC-ROC': [],
    'Training Time (min)': [],
    'Model Size (MB)': []
}

for model_name in available_models:
    if model_results[model_name] is not None:
        data = model_results[model_name]
        test_metrics = data.get('test_metrics', {})
        
        comparison_data['Model'].append(model_labels[model_name])
        comparison_data['Accuracy'].append(test_metrics.get('accuracy', 0))
        comparison_data['Precision'].append(test_metrics.get('precision', 0))
        comparison_data['Recall'].append(test_metrics.get('recall', 0))
        comparison_data['F1-Score'].append(test_metrics.get('f1_score', 0))
        comparison_data['AUC-ROC'].append(test_metrics.get('auc_roc', 0))
        
        # Training time
        training_time = (data.get('training_time_minutes') or
                         data.get('total_time_minutes') or
                         data.get('training_time', 0) / 60)
        comparison_data['Training Time (min)'].append(training_time)
        
        # Model size
        comparison_data['Model Size (MB)'].append(model_sizes[model_name])

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

if comparison_df.empty:
    print("❌ Tidak ada data untuk ditampilkan!")
    raise SystemExit("Data kosong")

print("\n📊 Tabel Perbandingan:")
print(comparison_df.round(4).to_string(index=False))

# Save comparison table
comparison_df.to_csv(comparison_dir / 'comparison_table.csv', index=False)
print(f"\n✅ Comparison table saved to: {comparison_dir / 'comparison_table.csv'}")

# ============================================
# 4. VISUALISASI PERBANDINGAN (Jika ada data)
# ============================================

print("\n[Visualization]")

# Set style
plt.style.use('seaborn-v0_8-darkgrid')

# Warna berdasarkan model yang tersedia
colors_used = [model_colors[m] for m in available_models]

# 4.1 Bar Chart - Accuracy, Precision, Recall, F1-Score
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = [
    ('Accuracy', comparison_df['Accuracy'], 0, 1.0),
    ('Precision', comparison_df['Precision'], 0, 1.0),
    ('Recall', comparison_df['Recall'], 0, 1.0),
    ('F1-Score', comparison_df['F1-Score'], 0, 1.0)
]

for i, (title, values, ymin, ymax) in enumerate(metrics_to_plot):
    row, col = i // 2, i % 2
    ax = axes[row, col]
    
    bars = ax.bar(comparison_df['Model'], values, color=colors_used)
    ax.set_title(f'Perbandingan {title}', fontsize=14, fontweight='bold')
    ax.set_ylim(ymin, ymax)
    ax.set_ylabel(title)
    ax.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(comparison_dir / 'comparison_metrics.png', dpi=300)
plt.show()

# 4.2 Bar Chart - Training Time & Model Size
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training Time
ax1 = axes[0]
bars1 = ax1.bar(comparison_df['Model'], comparison_df['Training Time (min)'], color=colors_used)
ax1.set_title('Perbandingan Waktu Training', fontsize=14, fontweight='bold')
ax1.set_ylabel('Waktu (menit)')
ax1.grid(True, alpha=0.3)
for bar, val in zip(bars1, comparison_df['Training Time (min)']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.1f} min', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Model Size
ax2 = axes[1]
bars2 = ax2.bar(comparison_df['Model'], comparison_df['Model Size (MB)'], color=colors_used)
ax2.set_title('Perbandingan Ukuran Model', fontsize=14, fontweight='bold')
ax2.set_ylabel('Ukuran (MB)')
ax2.grid(True, alpha=0.3)
for bar, val in zip(bars2, comparison_df['Model Size (MB)']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.0f} MB', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(comparison_dir / 'comparison_time_size.png', dpi=300)
plt.show()

# 4.3 Radar Chart (minimal 3 model untuk radar chart yang bagus)
if len(available_models) >= 2:
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))

    metrics_radar = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
    angles = np.linspace(0, 2 * np.pi, len(metrics_radar), endpoint=False).tolist()
    angles += angles[:1]  # Close the loop

    for i, model in enumerate(comparison_df['Model']):
        values = comparison_df[metrics_radar].iloc[i].tolist()
        values += values[:1]  # Close the loop
        
        ax.plot(angles, values, 'o-', linewidth=2, label=model, color=colors_used[i])
        ax.fill(angles, values, alpha=0.1, color=colors_used[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics_radar, fontsize=12)
    ax.set_ylim(0, 1.0)
    ax.set_title('Radar Chart Perbandingan Model', fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    ax.grid(True)

    plt.tight_layout()
    plt.savefig(comparison_dir / 'radar_chart.png', dpi=300)
    plt.show()

# ============================================
# 5. RINGKASAN STATISTIK
# ============================================

print("\n[Statistical Summary]")

if not comparison_df.empty:
    # Best model based on Accuracy
    best_accuracy = comparison_df.loc[comparison_df['Accuracy'].idxmax()]
    best_f1 = comparison_df.loc[comparison_df['F1-Score'].idxmax()]
    best_speed = comparison_df.loc[comparison_df['Training Time (min)'].idxmin()]
    best_size = comparison_df.loc[comparison_df['Model Size (MB)'].idxmin()]

    summary = {
        'Metrik': ['Akurasi Terbaik', 'F1-Score Terbaik', 'Training Tercepat', 'Model Teringan'],
        'Model': [best_accuracy['Model'], best_f1['Model'], best_speed['Model'], best_size['Model']],
        'Nilai': [
            f"{best_accuracy['Accuracy']*100:.2f}%",
            f"{best_f1['F1-Score']:.4f}",
            f"{best_speed['Training Time (min)']:.1f} min",
            f"{best_size['Model Size (MB)']:.0f} MB"
        ]
    }

    summary_df = pd.DataFrame(summary)
    print("\n📊 Ringkasan Pemenang:")
    print(summary_df.to_string(index=False))

    # Save summary
    summary_df.to_csv(comparison_dir / 'summary_best.csv', index=False)

# ============================================
# 6. CONFUSION MATRICES COMPARISON (Jika ada)
# ============================================

print("\n[Confusion Matrices Comparison]")

# Filter model yang punya confusion matrix
models_with_cm = []
for model_name in available_models:
    if model_results[model_name] is not None:
        cm = model_results[model_name].get('confusion_matrix', [])
        if cm:
            models_with_cm.append(model_name)

if models_with_cm:
    fig, axes = plt.subplots(1, len(models_with_cm), figsize=(5*len(models_with_cm), 5))
    if len(models_with_cm) == 1:
        axes = [axes]
    
    class_names = ['Awake', 'Drowsy']
    
    for idx, model_name in enumerate(models_with_cm):
        data = model_results[model_name]
        cm = data.get('confusion_matrix', [])
        cm_array = np.array(cm)
        
        ax = axes[idx]
        sns.heatmap(cm_array, annot=True, fmt='d', cmap='Blues',
                   xticklabels=class_names, yticklabels=class_names, ax=ax)
        ax.set_title(f'{model_labels[model_name]}', fontsize=12, fontweight='bold')
        ax.set_ylabel('Actual')
        ax.set_xlabel('Predicted')
        
        # Add accuracy text
        acc = comparison_df[comparison_df['Model'] == model_labels[model_name]]['Accuracy'].values[0]
        ax.text(0.5, -0.15, f'Accuracy: {acc*100:.2f}%', 
               ha='center', transform=ax.transAxes, fontsize=11, fontweight='bold')
    
    plt.suptitle('Perbandingan Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(comparison_dir / 'confusion_matrices_comparison.png', dpi=300)
    plt.show()
else:
    print("   ⚠️ Tidak ada confusion matrix yang tersedia")

# ============================================
# 7. RECOMMENDATION TABLE
# ============================================

print("\n[Recommendation]")

if not comparison_df.empty:
    # Find best model for each criteria
    best_acc = comparison_df.loc[comparison_df['Accuracy'].idxmax(), 'Model']
    best_f1 = comparison_df.loc[comparison_df['F1-Score'].idxmax(), 'Model']
    best_speed = comparison_df.loc[comparison_df['Training Time (min)'].idxmin(), 'Model']
    best_size = comparison_df.loc[comparison_df['Model Size (MB)'].idxmin(), 'Model']

    recommendations = {
        'Skenario': [
            'Aplikasi Mobile Real-time',
            'Akurasi Maksimal',
            'Edge Device (IoT)',
            'Cloud/Server',
            'Research Baseline'
        ],
        'Model Rekomendasi': [
            best_size if best_size in ['YOLOv8', 'MobileNetV2'] else 'YOLOv8',
            best_acc,
            best_size,
            best_acc,
            'CNN' if 'CNN' in comparison_df['Model'].values else 'MobileNetV2'
        ],
        'Alasan': [
            f'Ringan ({comparison_df[comparison_df["Model"]==best_size]["Model Size (MB)"].values[0]:.0f}MB) + Cepat',
            f'Akurasi tertinggi ({comparison_df[comparison_df["Model"]==best_acc]["Accuracy"].values[0]*100:.2f}%)',
            f'Ukuran kecil ({comparison_df[comparison_df["Model"]==best_size]["Model Size (MB)"].values[0]:.0f}MB)',
            f'Akurasi tinggi ({comparison_df[comparison_df["Model"]==best_acc]["Accuracy"].values[0]*100:.2f}%)',
            'Baseline untuk penelitian'
        ]
    }

    recommendation_df = pd.DataFrame(recommendations)
    print("\n📋 Rekomendasi Model Berdasarkan Skenario:")
    print(recommendation_df.to_string(index=False))
    recommendation_df.to_csv(comparison_dir / 'recommendations.csv', index=False)

# ============================================
# 8. FINAL SUMMARY
# ============================================

print("\n" + "="*60)
print("📊 EVALUASI SELESAI!")
print("="*60)

print(f"\n📁 Hasil evaluasi disimpan di: {comparison_dir}")

if not comparison_df.empty:
    print("\n📊 Ringkasan Final:")
    print("="*40)
    
    for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']:
        best_idx = comparison_df[metric].idxmax()
        print(f"   - {metric} Terbaik: {comparison_df.iloc[best_idx]['Model']} ({comparison_df.iloc[best_idx][metric]*100:.2f}%)")
    
    print("\n⏱️ Waktu Training Tercepat:")
    best_time_idx = comparison_df['Training Time (min)'].idxmin()
    print(f"   - {comparison_df.iloc[best_time_idx]['Model']} ({comparison_df.iloc[best_time_idx]['Training Time (min)']:.1f} menit)")
    
    print("\n💾 Model Teringan:")
    best_size_idx = comparison_df['Model Size (MB)'].idxmin()
    print(f"   - {comparison_df.iloc[best_size_idx]['Model']} ({comparison_df.iloc[best_size_idx]['Model Size (MB)']:.0f} MB)")

print("\n" + "="*60)
print("✅ SEMUA EVALUASI SELESAI!")
print("="*60)

print("\n📋 File yang dihasilkan:")
files = [
    'comparison_table.csv',
    'comparison_metrics.png',
    'comparison_time_size.png',
    'radar_chart.png',
    'confusion_matrices_comparison.png',
    'summary_best.csv',
    'recommendations.csv'
]
for f in files:
    file_path = comparison_dir / f
    if file_path.exists():
        print(f"   ✅ {f}")
    else:
        print(f"   ⚠️ {f} (not generated)")